# Scaled Dot-Product Attention Implementation

This notebook demonstrates the implementation of the scaled dot-product attention mechanism, a core component of Transformer models. It covers the generation of query (q), key (k), and value [...]

In [5]:
import numpy as np
import math

L,d_k,d_v = 4,8,8
q = np.random.randn(L,d_k)
k = np.random.randn(L,d_k)
v = np.random.randn(L,d_v)

In [6]:
print("Q:", q)
print("K:",k)
print("V",v)

Q: [[-1.25178099e+00  6.73986443e-01  1.53201254e+00 -1.77525385e-01
   4.89971655e-01  1.78084742e+00  6.83328409e-01 -2.02483056e-04]
 [-3.58243516e-01 -7.43565211e-01  7.24590695e-01  1.56610061e+00
  -8.15638641e-01  3.24346753e-01  4.53051720e-01 -9.85585187e-01]
 [ 7.19713301e-01 -1.23393983e+00  5.60528845e-01  4.78980478e-01
   7.24480298e-01  3.25591878e-01  1.31860840e+00  9.42466247e-02]
 [-1.19391794e+00  1.53026053e+00 -9.52317071e-01 -4.59714885e-01
   1.35142517e+00  8.97449106e-01 -2.51202711e+00 -7.99271578e-01]]
K: [[-2.5570479  -0.31390546  0.77161968 -0.70239545  0.60637802  0.91055047
  -0.06719873  0.52805558]
 [-0.92677208 -0.86257575 -1.80961994 -0.52195894 -1.62008209 -2.07830995
   0.55609003 -1.51305472]
 [ 0.02797675 -0.15559345 -0.97371112  0.09154415 -1.29167371  0.90675897
   0.25961263 -0.17058845]
 [-0.1504562  -0.05144002  0.24646146  0.59545456 -0.02342938 -0.55262563
  -0.56760893 -0.47710133]]
V [[-2.1887267  -0.52772172 -0.81408604  1.17034636  0.4

In [7]:
np.matmul(q,k.T)

array([[ 6.16875374, -6.21559689, -0.48852661, -0.95784629],
       [-0.14159801,  1.23520776,  1.17688743,  1.25620978],
       [-0.65998415, -2.12672964, -0.6041259 , -0.61177521],
       [ 3.54401572, -2.49236029, -0.83394075,  0.87203306]])

In [8]:
q.var(),k.var(),np.matmul(q,k.T).var()

(np.float64(1.019043313446817),
 np.float64(0.7608822677597082),
 np.float64(6.764342511923852))

In [9]:
scaled = np.matmul(q,k.T)/math.sqrt(d_k)
q.var(),k.var(),scaled.var()

(np.float64(1.019043313446817),
 np.float64(0.7608822677597082),
 np.float64(0.8455428139904815))

In [11]:
mask = np.tril(np.ones((L,L)))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [15]:
mask[mask==0] = -np.inf
mask[mask==1] = 0

In [16]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [17]:
scaled+mask

array([[ 2.1809838 ,        -inf,        -inf,        -inf],
       [-0.05006246,  0.43671189,        -inf,        -inf],
       [-0.23333964, -0.75191248, -0.21359076,        -inf],
       [ 1.25299877, -0.88118243, -0.29484258,  0.30831025]])

In [18]:
def softmax(x):
  return (np.exp(x).T/np.sum(np.exp(x),axis=-1)).T

The `softmax` function is defined as:

$$ \text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}} $$

This function converts a vector of numbers into a probability distribution, where the sum of the output elements is 1.

In [19]:
attention = softmax(scaled+mask)

The attention weights are calculated by applying the softmax function to the scaled dot product of the query and key matrices, possibly after adding a mask:

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right)V $$

Where $M$ is the optional mask (e.g., for masked self-attention).

In [20]:
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.38065374, 0.61934626, 0.        , 0.        ],
       [0.38236315, 0.22764741, 0.38998945, 0.        ],
       [0.58144648, 0.06880922, 0.12367753, 0.22606677]])

In [21]:
new_v = np.matmul(attention,v)
new_v

array([[-2.1887267 , -0.52772172, -0.81408604,  1.17034636,  0.46253979,
         1.07939775,  1.82316015, -0.27119805],
       [-0.93775288, -0.1893667 , -0.91461236,  0.52910961,  1.54889247,
         1.36167567,  1.01584853,  0.86066566],
       [-1.24250684, -0.49639976, -0.14229966,  0.54092706,  1.4770557 ,
         0.46120163,  0.62075761,  0.45499562],
       [-1.45202915, -0.30838326, -0.64631336,  0.88055125,  0.84191161,
         0.6027514 ,  0.86433812, -0.09602403]])

In [22]:
v

array([[-2.1887267 , -0.52772172, -0.81408604,  1.17034636,  0.46253979,
         1.07939775,  1.82316015, -0.27119805],
       [-0.16889724,  0.01858822, -0.97639642,  0.13500184,  2.21657103,
         1.53516529,  0.5196702 ,  1.55631556],
       [-0.94148545, -0.76630273,  1.00323472,  0.1607657 ,  2.04005749,
        -0.77180726, -0.49912431,  0.52411819],
       [-0.22709852,  0.40675621, -1.01677221,  0.75590499,  0.74376122,
        -0.15499372, -0.75092672, -0.48767617]])

In [23]:
def softmax(x):
  return (np.exp(x).T/np.sum(np.exp(x),axis=-1)).T

def scaled_dot_product_attention(q,k,v,mask=None):
  d_k = q.shape[-1]
  scaled = np.matmul(q,k.T)/math.sqrt(d_k)
  if mask is not None:
    scaled = scaled+mask
  attention = softmax(scaled)
  output = np.matmul(attention,v)
  return output,attention

In [24]:
values,attention = scaled_dot_product_attention(q,k,v,mask=mask)
print("Q: ",q)
print("K: ",k)
print("V: ",v)
print("New V: ",values)
print("Attention: ",attention)

Q:  [[-1.25178099e+00  6.73986443e-01  1.53201254e+00 -1.77525385e-01
   4.89971655e-01  1.78084742e+00  6.83328409e-01 -2.02483056e-04]
 [-3.58243516e-01 -7.43565211e-01  7.24590695e-01  1.56610061e+00
  -8.15638641e-01  3.24346753e-01  4.53051720e-01 -9.85585187e-01]
 [ 7.19713301e-01 -1.23393983e+00  5.60528845e-01  4.78980478e-01
   7.24480298e-01  3.25591878e-01  1.31860840e+00  9.42466247e-02]
 [-1.19391794e+00  1.53026053e+00 -9.52317071e-01 -4.59714885e-01
   1.35142517e+00  8.97449106e-01 -2.51202711e+00 -7.99271578e-01]]
K:  [[-2.5570479  -0.31390546  0.77161968 -0.70239545  0.60637802  0.91055047
  -0.06719873  0.52805558]
 [-0.92677208 -0.86257575 -1.80961994 -0.52195894 -1.62008209 -2.07830995
   0.55609003 -1.51305472]
 [ 0.02797675 -0.15559345 -0.97371112  0.09154415 -1.29167371  0.90675897
   0.25961263 -0.17058845]
 [-0.1504562  -0.05144002  0.24646146  0.59545456 -0.02342938 -0.55262563
  -0.56760893 -0.47710133]]
V:  [[-2.1887267  -0.52772172 -0.81408604  1.17034636 